# NPCR: Neural Pairwise Contrastive Regression for AES implementation by Psychometrics.ai

Reimplementation of Xie, Cai, Kong, Zhou, Qu (2022), *Automated Essay Scoring via Pairwise Contrastive Regression*, COLING 2022.

5-fold cross-validation (§4.1.1), prompt 4. `hidden_dim` 768, flat LR (no schedule), `max_length` 512, batch size 5, dropout(0.5), accumulation pairing, literal `[CLS]`.

**Sections**
1. Setup & data loading
2. 5-fold cross-validation splits (§4.1.1)
3. Text → BERT `[CLS]` embedding
4. Neural pairwise ranker (§3.2)
5. Pairwise contrastive regression head (§3.3)
6. Training pair construction incl. accumulation (§3.3)
7. Training loop, wrapped for 5-fold CV (`run_fold`)
8. Multi-sample voting inference (§3.4) -- embedded in `run_fold`
9. Evaluation: 5-fold average Quadratic Weighted Kappa (§4.1.2, §4.1.1)
10. Ablations (stretch goals, §4.3)


## 1. Setup & data loading

In [1]:
!pip install -q transformers  # harmless if already installed (e.g. locally); makes sure it's there on Colab

import pandas as pd
import numpy as np
import torch

# on Colab: mount Drive and switch into this project's synced folder so the relative paths below
# (training_set_rel3.tsv, etc.) resolve the same way they do locally -- no manual upload needed
try:
    from google.colab import drive
    drive.mount("/content/drive")
    import os
    os.chdir("/content/drive/MyDrive/01 NCPR")
except ImportError:
    pass  # not on Colab -- already sitting in the right folder locally

print(torch.__version__, torch.cuda.is_available(), torch.backends.mps.is_available())

Mounted at /content/drive
2.11.0+cu128 True False


In [2]:
# Kaggle ASAP-AES release. Encoding is latin-1 (essays contain non-UTF8 bytes).
# Data file lives next to this notebook (not in asap-aes/, which holds material we don't share).
df = pd.read_csv("training_set_rel3.tsv", sep="\t", encoding="latin-1")

# prompt 4 only: score range 0-3, ~1770 essays
df = df[df.essay_set == 4][["essay_id", "essay_set", "essay", "domain1_score"]].reset_index(drop=True)

REL_RANGE = df.domain1_score.max() - df.domain1_score.min()  # 0-3 for prompt 4
SCORE_MIN, SCORE_MAX = df.domain1_score.min(), df.domain1_score.max()

len(df), SCORE_MIN, SCORE_MAX

(1770, 0, 3)

## 2. 5-fold cross-validation splits (§4.1.1)

5-fold CV: rotate which fold plays test/valid/train across 5 runs. Reported `0.851` is the average of 5 test scores.

| run | test | valid | train |
|---|---|---|---|
| 1 | fold 1 | fold 2 | folds 3+4+5 |
| 2 | fold 2 | fold 3 | folds 1+4+5 |
| 3 | fold 3 | fold 4 | folds 1+2+5 |
| 4 | fold 4 | fold 5 | folds 1+2+3 |
| 5 | fold 5 | fold 1 | folds 2+3+4 |

BERT resets to pretrained weights at the start of each fold (Section 7).


In [3]:
# 5 folds of ~354 essays each, shuffled once with a fixed seed for reproducibility.
# last fold absorbs the remainder if essay count doesn't divide evenly.
shuffled = df.sample(frac=1, random_state=0).reset_index(drop=True)
fold_size = len(shuffled) // 5
folds = [shuffled[i * fold_size:(i + 1) * fold_size].reset_index(drop=True) for i in range(5)]
folds[4] = shuffled[4 * fold_size:].reset_index(drop=True)

[len(f) for f in folds]

[354, 354, 354, 354, 354]

## 3. Text → BERT `[CLS]` embedding

Eq. 5–6: `h = BERT(e)`, `f = h[CLS]`.

`bert`/`tokenizer`/`embed()` below demo the mechanics only. Section 7 loads a fresh `bert` per fold.



In [4]:
# Load BERT-base and its tokenizer once; reuse for every essay (paper §4.1.3 uses BERT-base).
from transformers import AutoTokenizer, AutoModel

# cuda > mps > cpu -- same notebook runs unmodified on a cloud GPU or this Mac
device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
bert = AutoModel.from_pretrained("bert-base-uncased").to(device).eval()
bert.gradient_checkpointing_enable()  # recompute activations in backward instead of storing them --
# real memory savings, needed after hitting MPS OOM on the Mac. costs some speed either way.

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [5]:
# Eq. 5-6: h = BERT(e), f = h[CLS]. 512-token truncation.
# no torch.no_grad() -- Section 7 fine-tunes BERT end-to-end, gradients need to flow through this.
def embed(text):
    tokens = tokenizer(text, truncation=True, max_length=512, return_tensors="pt").to(device)
    h = bert(**tokens).last_hidden_state
    return h[0, 0]  # the [CLS] token's vector, f

In [6]:
# a short example, small enough that the two stages stay readable
demo = tokenizer("The cat sat on the mat.", return_tensors="pt").to(device)
with torch.no_grad():
    demo_h = bert(**demo).last_hidden_state

print("stage 1 -- one row per token, shape:", tuple(demo_h.shape))  # (1, num_tokens, 768)
print(tokenizer.convert_ids_to_tokens(demo["input_ids"][0]))
print(demo_h[0, :, :5])  # first 5 of 768 dims, every token

demo_f = demo_h[0, 0]  # keep only the [CLS] row -- this is Eq. 6
print("\nstage 2 -- [CLS] row only, shape:", tuple(demo_f.shape))
print(demo_f[:5])  # same first 5 dims, just the one kept row

stage 1 -- one row per token, shape: (1, 9, 768)
['[CLS]', 'the', 'cat', 'sat', 'on', 'the', 'mat', '.', '[SEP]']
tensor([[-0.3642, -0.0531, -0.3673, -0.0297, -0.4608],
        [-0.3979, -0.2721, -0.6820, -0.0073,  0.7860],
        [-0.3512, -0.0736, -0.0691, -0.1399,  0.6829],
        [ 0.0712, -0.3137,  0.0980,  0.0693,  0.4834],
        [-0.5204, -0.5930,  0.2836,  0.3123,  0.6113],
        [-0.4620, -0.5198, -0.3760,  0.5099,  0.4772],
        [-0.0415, -0.1055, -0.2808,  0.5945,  0.0549],
        [-0.2354, -0.4875, -0.1631,  0.2472,  0.1660],
        [ 0.6651,  0.0225, -0.4131,  0.3417, -0.2384]], device='cuda:0')

stage 2 -- [CLS] row only, shape: (768,)
tensor([-0.3642, -0.0531, -0.3673, -0.0297, -0.4608], device='cuda:0')


In [7]:
# one essay in, one 768-number vector out
f = embed(df.essay[0])
f.shape

# paper writes essay embeddings as dims x essays, we'll stack ours as essays x dims
# (one row per essay) -- same numbers, table just laid out sideways from the paper

torch.Size([768])

## 4. Neural pairwise ranker (§3.2)

Weight-shared MLP subnets `nn1`, `nn2` → difference vector `dv = Fw(f1) - Fw(f2)` (Eq. 7) → `nn3` with antisymmetric activation and no bias (Eq. 8–10).

`Fw` ends in `nn.Dropout(0.5)`. `hidden_dim` is 768.

Class definition shared/reused; a fresh instance is created per fold in Section 7.


In [8]:
import torch.nn as nn

class NPCRModel(nn.Module):
    def __init__(self, embed_dim=768, hidden_dim=768):
        super().__init__()
        self.Fw = nn.Sequential(nn.Linear(embed_dim, hidden_dim), nn.ReLU(), nn.Dropout(0.5))
        self.nn3 = nn.Linear(hidden_dim, 1, bias=False)  # no bias -- required for antisymmetry

    def forward(self, f1, f2):
        dv = self.Fw(f1) - self.Fw(f2)   # Eq. 7
        return torch.tanh(self.nn3(dv))  # tanh(-z) = -tanh(z) -- required for antisymmetry

torch.manual_seed(0)  # so the random weights below are reproducible
model = NPCRModel().to(device)  # same device as embed()'s output, or forward() can't run on it


In [9]:
# checks the architecture itself, using random essay-sized vectors and untrained weights.
# if rf(x,y) == -rf(y,x) holds here, the wiring guarantees it, before any training happens.
# eval() is required: Fw's dropout is stochastic, so in train mode model(x,y) and
# model(y,x) sample different random masks for Fw(x) and Fw(y) -- the antisymmetry proof only
# holds when Fw is deterministic, which is only true with dropout off (eval mode).
model.eval()
x, y = torch.randn(1, 768).to(device), torch.randn(1, 768).to(device)

rf_xy = model(x, y).item()
rf_yx = model(y, x).item()

print(f"rf(x,y) = {rf_xy:.6f}")
print(f"rf(y,x) = {rf_yx:.6f}")
print(f"antisymmetric? {abs(rf_xy + rf_yx) < 1e-6}")

rf(x,y) = -0.380419
rf(y,x) = 0.380419
antisymmetric? True


## 5. Pairwise contrastive regression head (§3.3)

`nn3` outputs a real-valued relative score `Δs = Rθ(dv)` (Eq. 11), trained with MSE against gold relative scores (Eq. 12).

`predict_score(f_i, f_j, s_j)` adds a reference essay's score to the model's output (Eq. 3). The training target (`delta_s_true`, Section 7) is normalized by `REL_RANGE` so it lands in tanh's native (-1,1) range -- the model's raw output is on that same normalized scale, not raw score points, so it has to be multiplied back by `REL_RANGE` before adding to `s_j`. Skipping that step caps every correction at +/-1 score point regardless of how large the true difference actually is, which compresses every prediction toward the reference essay's own score. The version used during CV training is defined inside `run_fold` (Section 7).


In [10]:
# Eq. 3/11: turn the model's relative judgement into an absolute score guess.
# model's output is on the normalized (-1,1) scale (see Section 7) -- multiply back by REL_RANGE
# to get raw score points before adding to s_j, a plain gold label (no tensor, no gradient here).
def predict_score(f_i, f_j, s_j):
    delta_s = model(f_i, f_j).item() * REL_RANGE
    return delta_s + s_j


In [11]:
# smoke test on two real essays -- model is untrained, so this is expected to be a bad guess.
# only checking the plumbing works here -- the prediction quality comes later, once trained.
# uses df directly -- train_df only exists inside run_fold
essay_i, essay_j = df.iloc[0], df.iloc[1]
f_i, f_j = embed(essay_i.essay), embed(essay_j.essay)

predicted = predict_score(f_i, f_j, essay_j.domain1_score)
print(f"essay i's true score: {essay_i.domain1_score}, predicted (untrained): {predicted:.3f}")

essay i's true score: 0, predicted (untrained): -0.207


## 6. Training pair construction incl. accumulation (§3.3)

Adjacent pairs, plus accumulation: if `(ei, ej)` and `(ej, ek)` are pairs, also add `(ei, ek)`.

One pass only. `2n-3` pairs for n essays, under the paper's `<2n` bound (§4.3.6).


In [12]:
def build_pairs(train_df):  # pairs are (i, j) positions into train_df -- embeddings get looked up later
    n = len(train_df)
    adjacent_pairs = [(i, i + 1) for i in range(n - 1)]
    accumulated_pairs = [(i, i + 2) for i in range(n - 2)]  # from (i,i+1) + (i+1,i+2)
    return adjacent_pairs + accumulated_pairs

# check against the paper's cost bound (§4.3.6), using all 1770 essays as a stand-in for a train_df
demo_pairs = build_pairs(df)
n = len(df)
print(f"n = {n} -> {len(demo_pairs)} pairs (expect 2n-3 = {2*n-3}), under 2n? {len(demo_pairs) < 2*n}")

n = 1770 -> 3537 pairs (expect 2n-3 = 3537), under 2n? True


## 7. Training loop, wrapped for 5-fold CV

§4.1.3: BERT-base, 512 tokens, batch size 5, 80 epochs, AdamW, lr 1e-5.

`run_fold(fold_id, epochs)`: training + per-epoch validation/checkpointing + multi-sample voting, run once per fold with fresh weights.

- Fine-tunes BERT end-to-end.
- Normalizes to (-1,1), not [0,1] -- reflexivity forces rf(x,x)=0, so predictions get trained and stored on this normalized scale, then multiplied back by `REL_RANGE` at inference (Section 5).
- Flat LR, no schedule, no gradient clipping -- matches the reference implementation's actual training loop, not just an inference from the paper's wording.
- `epochs=20`, checkpointed so more epochs can only find an equal-or-better result.
- Batch size 5, matching the paper's stated value (§4.1.3).
- Dropout(0.5) inside `Fw`.
- `max_length=512`.
- Epoch-level resume-on-restart from `fold_{fold_id}_best.pt`, including optimizer state (momentum/variance), not just model weights.
- Mid-epoch checkpointing to `fold_{fold_id}_progress.pt` every `SAVE_EVERY` steps.

`embed`, `predict_score`, `score_essay` are closures inside `run_fold` over that fold's own `bert`/`model`.


In [13]:
from sklearn.metrics import cohen_kappa_score
import random, math, os

M = 40  # reference essays for multi-sample voting, paper's plateau point
BATCH_SIZE = 5  # paper's stated batch size (§4.1.3)
PRINT_EVERY = 20  # batches between progress prints
SAVE_EVERY = 60   # batches between mid-epoch checkpoint saves

def clipped_round(x):
    return int(max(SCORE_MIN, min(SCORE_MAX, round(x))))

def run_fold(fold_id, epochs=20):
    test_df = folds[fold_id]
    valid_df = folds[(fold_id + 1) % 5]
    train_parts = [folds[i] for i in range(5) if i not in (fold_id, (fold_id + 1) % 5)]
    train_df = pd.concat(train_parts).reset_index(drop=True)

    # fresh BERT + model every fold, so each fold trains independently of the last
    fold_bert = AutoModel.from_pretrained("bert-base-uncased").to(device)
    fold_bert.gradient_checkpointing_enable()
    torch.manual_seed(0)
    fold_model = NPCRModel().to(device)

    def embed(text):
        tokens = tokenizer(text, truncation=True, max_length=512, return_tensors="pt").to(device)
        return fold_bert(**tokens).last_hidden_state[0, 0]  # [CLS]

    def embed_batch(texts):
        tokens = tokenizer(texts, truncation=True, max_length=512, padding=True, return_tensors="pt").to(device)
        return fold_bert(**tokens).last_hidden_state[:, 0]  # [CLS]; padding is attention-masked

    def predict_score(f_i, f_j, s_j):  # Eq. 3/11, closes over this fold's model
        return fold_model(f_i, f_j).item() * REL_RANGE + s_j

    def score_essay(text, ref_embeddings):  # Eq. 14-15
        with torch.no_grad():
            f_test = embed(text)
            preds = [predict_score(f_test, f_ref, s_ref) for f_ref, s_ref in ref_embeddings]
        return sum(preds) / len(ref_embeddings)

    pairs = build_pairs(train_df)
    refs = train_df.sample(M, random_state=0)

    best_path = f"fold_{fold_id}_best.pt"
    progress_path = f"fold_{fold_id}_progress.pt"

    optimizer = torch.optim.AdamW(list(fold_bert.parameters()) + list(fold_model.parameters()), lr=1e-5)

    if os.path.exists(progress_path):
        ckpt, source = torch.load(progress_path, map_location=device), "progress"
    elif os.path.exists(best_path):
        ckpt, source = torch.load(best_path, map_location=device), "best"
    else:
        ckpt = None

    if ckpt is not None:
        fold_bert.load_state_dict(ckpt["bert"])
        fold_model.load_state_dict(ckpt["model"])
        if "optimizer" in ckpt:
            optimizer.load_state_dict(ckpt["optimizer"])
        start_epoch, best_qwk = ckpt["epoch"], ckpt["qwk"]
        print(f"  fold {fold_id}: resuming from {source} checkpoint (epoch {start_epoch}, QWK={best_qwk:.3f})")
    else:
        start_epoch, best_qwk = 0, -1.0

    fold_bert.train()
    fold_model.train()

    random.seed(0)

    for epoch in range(start_epoch, epochs):
        fold_bert.train()
        fold_model.train()
        epoch_pairs = pairs.copy()
        random.shuffle(epoch_pairs)
        n_batches = math.ceil(len(epoch_pairs) / BATCH_SIZE)

        running_loss = 0.0
        pairs_seen = 0
        for step in range(1, n_batches + 1):
            batch = epoch_pairs[(step - 1) * BATCH_SIZE : step * BATCH_SIZE]
            essays_i = [train_df.iloc[i] for i, j in batch]
            essays_j = [train_df.iloc[j] for i, j in batch]
            embeddings = embed_batch([e.essay for e in essays_i] + [e.essay for e in essays_j])
            f_i, f_j = embeddings[:len(batch)], embeddings[len(batch):]

            delta_s_true = torch.tensor([(ei.domain1_score - ej.domain1_score) / REL_RANGE
                                          for ei, ej in zip(essays_i, essays_j)],
                                         dtype=torch.float32, device=device).unsqueeze(1)
            delta_s_pred = fold_model(f_i, f_j)
            loss = ((delta_s_pred - delta_s_true) ** 2).mean()

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * len(batch)
            pairs_seen += len(batch)
            if step % 10 == 0:
                if device == "mps":
                    torch.mps.empty_cache()
                elif device == "cuda":
                    torch.cuda.empty_cache()

            if step % PRINT_EVERY == 0:
                print(f"    fold {fold_id}, epoch {epoch+1}/{epochs}, batch {step}/{n_batches}: running avg loss={running_loss/pairs_seen:.4f}")

            if step % SAVE_EVERY == 0:
                torch.save({"bert": fold_bert.state_dict(), "model": fold_model.state_dict(),
                            "optimizer": optimizer.state_dict(),
                            "epoch": epoch, "qwk": float(best_qwk)}, progress_path)

        fold_bert.eval()
        fold_model.eval()
        with torch.no_grad():
            ref_embeddings = [(embed(row.essay), row.domain1_score) for _, row in refs.iterrows()]
        valid_preds = [clipped_round(score_essay(row.essay, ref_embeddings)) for _, row in valid_df.iterrows()]
        valid_true = valid_df.domain1_score.tolist()
        epoch_qwk = cohen_kappa_score(valid_true, valid_preds, weights="quadratic")

        is_best = epoch_qwk > best_qwk
        if is_best:
            best_qwk = epoch_qwk
            torch.save({"bert": fold_bert.state_dict(), "model": fold_model.state_dict(),
                        "optimizer": optimizer.state_dict(),
                        "epoch": epoch + 1, "qwk": float(epoch_qwk)}, best_path)

        marker = " <- new best" if is_best else ""
        print(f"  fold {fold_id}, epoch {epoch+1}/{epochs}: avg loss={running_loss/pairs_seen:.4f}, valid QWK={epoch_qwk:.3f}{marker}")

    # load this fold's best checkpoint, evaluate on its held-out test set
    ckpt = torch.load(best_path, map_location=device)
    fold_bert.load_state_dict(ckpt["bert"])
    fold_model.load_state_dict(ckpt["model"])
    fold_bert.eval()
    fold_model.eval()
    with torch.no_grad():
        ref_embeddings = [(embed(row.essay), row.domain1_score) for _, row in refs.iterrows()]
    test_preds = [clipped_round(score_essay(row.essay, ref_embeddings)) for _, row in test_df.iterrows()]
    test_true = test_df.domain1_score.tolist()
    test_qwk = cohen_kappa_score(test_true, test_preds, weights="quadratic")

    os.remove(best_path)
    if os.path.exists(progress_path):
        os.remove(progress_path)
    print(f"fold {fold_id} done: best valid QWK={best_qwk:.3f}, test QWK={test_qwk:.3f}")
    return test_qwk


In [14]:
# smoke test retired -- disabled, since "Run all" on a resumed run could delete real progress

# smoke_qwk = run_fold(fold_id=0, epochs=1)
# print(f"smoke test fold 0 test QWK (1 epoch, not meaningful): {smoke_qwk:.3f}")


## 8. Multi-sample voting inference (§3.4)

Eq. 14–15: pair the test essay with `M` reference essays, average the `M` predicted scores.

Embedded inside `run_fold` (`score_essay`, `M`, `refs`).


## 9. Evaluation: 5-fold average Quadratic Weighted Kappa (§4.1.2, §4.1.1)

Eq. 16–17 via `sklearn.metrics.cohen_kappa_score`. Average of 5 independent test QWK scores -- comparable to paper Table 2's `0.851` for prompt 4.


In [15]:
# all 5 folds, 20 epochs each. skips any fold that already has a saved result on disk.
fold_test_qwks = []
for fold_id in range(5):
    result_path = f"fold_{fold_id}_result.txt"
    if os.path.exists(result_path):
        test_qwk = float(open(result_path).read())
        print(f"=== fold {fold_id} already done, test QWK={test_qwk:.3f} (skipping) ===")
    else:
        print(f"=== fold {fold_id} ===")
        test_qwk = run_fold(fold_id, epochs=20)
        with open(result_path, "w") as f:
            f.write(str(test_qwk))
    fold_test_qwks.append(test_qwk)

fold_test_qwks


=== fold 0 ===


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
[transformers] `use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


    fold 0, epoch 1/20, batch 20/425: running avg loss=0.1983
    fold 0, epoch 1/20, batch 40/425: running avg loss=0.1822
    fold 0, epoch 1/20, batch 60/425: running avg loss=0.1554
    fold 0, epoch 1/20, batch 80/425: running avg loss=0.1429
    fold 0, epoch 1/20, batch 100/425: running avg loss=0.1304
    fold 0, epoch 1/20, batch 120/425: running avg loss=0.1228
    fold 0, epoch 1/20, batch 140/425: running avg loss=0.1199
    fold 0, epoch 1/20, batch 160/425: running avg loss=0.1155
    fold 0, epoch 1/20, batch 180/425: running avg loss=0.1155
    fold 0, epoch 1/20, batch 200/425: running avg loss=0.1118
    fold 0, epoch 1/20, batch 220/425: running avg loss=0.1087
    fold 0, epoch 1/20, batch 240/425: running avg loss=0.1055
    fold 0, epoch 1/20, batch 260/425: running avg loss=0.1035
    fold 0, epoch 1/20, batch 280/425: running avg loss=0.1003
    fold 0, epoch 1/20, batch 300/425: running avg loss=0.0980
    fold 0, epoch 1/20, batch 320/425: running avg loss=0.0

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

...

    fold 3, epoch 1/20, batch 20/425: running avg loss=0.1786
    fold 3, epoch 1/20, batch 40/425: running avg loss=0.1502
    fold 3, epoch 1/20, batch 60/425: running avg loss=0.1379
    fold 3, epoch 1/20, batch 80/425: running avg loss=0.1342
    fold 3, epoch 1/20, batch 100/425: running avg loss=0.1329
    fold 3, epoch 1/20, batch 120/425: running avg loss=0.1264
    fold 3, epoch 1/20, batch 140/425: running avg loss=0.1197
    fold 3, epoch 1/20, batch 160/425: running avg loss=0.1138
    fold 3, epoch 1/20, batch 180/425: running avg loss=0.1101
    fold 3, epoch 1/20, batch 200/425: running avg loss=0.1082
    fold 3, epoch 1/20, batch 220/425: running avg loss=0.1045
    fold 3, epoch 1/20, batch 240/425: running avg loss=0.1002
    fold 3, epoch 1/20, batch 260/425: running avg loss=0.0978
    fold 3, epoch 1/20, batch 280/425: running avg loss=0.0944
    fold 3, epoch 1/20, batch 300/425: running avg loss=0.0915
    fold 3, epoch 1/20, batch 320/425: running avg loss=0.0

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


    fold 4, epoch 1/20, batch 20/425: running avg loss=0.1729
    fold 4, epoch 1/20, batch 40/425: running avg loss=0.1585
    fold 4, epoch 1/20, batch 60/425: running avg loss=0.1420
    fold 4, epoch 1/20, batch 80/425: running avg loss=0.1317
    fold 4, epoch 1/20, batch 100/425: running avg loss=0.1225
    fold 4, epoch 1/20, batch 120/425: running avg loss=0.1165
    fold 4, epoch 1/20, batch 140/425: running avg loss=0.1111
    fold 4, epoch 1/20, batch 160/425: running avg loss=0.1082
    fold 4, epoch 1/20, batch 180/425: running avg loss=0.1041
    fold 4, epoch 1/20, batch 200/425: running avg loss=0.1023
    fold 4, epoch 1/20, batch 220/425: running avg loss=0.0997
    fold 4, epoch 1/20, batch 240/425: running avg loss=0.0965
    fold 4, epoch 1/20, batch 260/425: running avg loss=0.0948
    fold 4, epoch 1/20, batch 280/425: running avg loss=0.0937
    fold 4, epoch 1/20, batch 300/425: running avg loss=0.0924
    fold 4, epoch 1/20, batch 320/425: running avg loss=0.0

[np.float64(0.8362718990889979),
 np.float64(0.8184615384615385),
 np.float64(0.8130693069306931),
 np.float64(0.8108696291503691),
 np.float64(0.8151943948344553)]

In [16]:
# the number that's actually comparable to the paper's Table 2 (0.851 for prompt 4)
cv_average_qwk = sum(fold_test_qwks) / len(fold_test_qwks)
print(f"per-fold test QWK: {[round(q, 3) for q in fold_test_qwks]}")
print(f"5-fold CV average QWK: {cv_average_qwk:.3f}  (paper's prompt-4 QWK: 0.851)")

per-fold test QWK: [np.float64(0.836), np.float64(0.818), np.float64(0.813), np.float64(0.811), np.float64(0.815)]
5-fold CV average QWK: 0.819  (paper's prompt-4 QWK: 0.851)


## 10. Ablations (stretch goals, §4.3)

Only pursue once the core pipeline above works end-to-end:
- Regression-only vs. rank-only vs. full NPCR (Table 3)
- Accumulation vs. no-accumulation pair selection (Table 4)
- Effect of `M` in multi-sample voting (Fig. 2)
- Alternate backbones: BERT vs. XLNet vs. RoBERTa (Table 5)